In [1]:
import pandas as pd

# We use 2017 as the template to grab the exact column headers
reference_file = "anon-Booter_Combined.csv" 
output_file = "leave_BoT-IoT.csv"

# Read only headers
df_ref = pd.read_csv(reference_file, nrows=0)

# Rename the 3 culprit columns to match the target schema
df_ref = df_ref.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

# Create the empty master file
df_ref.to_csv(output_file, index=False)
print(f"Master schema created: {output_file}")

Master schema created: leave_BoT-IoT.csv


In [2]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "caida_dataset_2007_Combined.csv"
current_dataset_name = "caida"

big_file = "leave_BoT-IoT.csv"
n = 5933   # attack total samples
m = 0   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CAIDA
--------------------------------------------------
Original row count: 2471740
Auto-detected 1 distinct attack class(es).
Targeting 5933 records per class.

SUCCESS! Summary:
  -> Benign Rows: 0 (m-based)
  -> Attack Rows: 5933 (n-based)
  -> Total Appended: 5933 rows
--------------------------------------------------


In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "anon-Booter_Combined.csv"
current_dataset_name = "booters"

big_file = "leave_BoT-IoT.csv"
n = 5933   # attack total samples
m = 0   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: BOOTERS
--------------------------------------------------
Original row count: 1771055
Auto-detected 1 distinct attack class(es).
Targeting 5933 records per class.

SUCCESS! Summary:
  -> Benign Rows: 0 (m-based)
  -> Attack Rows: 5933 (n-based)
  -> Total Appended: 5933 rows
--------------------------------------------------


In [2]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "CIC-IDS-2017_Binary.csv"
current_dataset_name = "cic17"

big_file = "leave_BoT-IoT.csv"
n = 5933   # attack total samples
m = 11866   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CIC17
--------------------------------------------------
Original row count: 2830743
Auto-detected 1 distinct attack class(es).
Targeting 5933 records per class.

SUCCESS! Summary:
  -> Benign Rows: 11866 (m-based)
  -> Attack Rows: 5933 (n-based)
  -> Total Appended: 17799 rows
--------------------------------------------------


In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "CSC_CIC_02-20-2018.csv"
current_dataset_name = "cic18"

big_file = "leave_BoT-IoT.csv"
n = 5933   # attack total samples
m = 11866   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CIC18
--------------------------------------------------
Original row count: 7948748
Auto-detected 1 distinct attack class(es).
Targeting 5933 records per class.

SUCCESS! Summary:
  -> Benign Rows: 11866 (m-based)
  -> Attack Rows: 5933 (n-based)
  -> Total Appended: 17799 rows
--------------------------------------------------


In [2]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "CTU-13.csv"
current_dataset_name = "ctu13"

big_file = "leave_BoT-IoT.csv"
n = 5933   # attack total samples
m = 0   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CTU13
--------------------------------------------------
Original row count: 5933
Auto-detected 1 distinct attack class(es).
Targeting 5933 records per class.

SUCCESS! Summary:
  -> Benign Rows: 0 (m-based)
  -> Attack Rows: 5933 (n-based)
  -> Total Appended: 5933 rows
--------------------------------------------------


In [3]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "part1CIC2019.csv"
current_dataset_name = "cic19"

big_file = "leave_BoT-IoT.csv"
n = 2967   # attack total samples
m = 5933   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CIC19
--------------------------------------------------
Original row count: 25031556
Auto-detected 8 distinct attack class(es).
Targeting 370 records per class.

SUCCESS! Summary:
  -> Benign Rows: 5933 (m-based)
  -> Attack Rows: 2960 (n-based)
  -> Total Appended: 8893 rows
--------------------------------------------------


In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. CONFIGURATION
# ==========================================
current_file_path = "part2CIC2019.csv"
current_dataset_name = "cic19"

big_file = "leave_BoT-IoT.csv"
n = 2966   # attack total samples
m = 5933   # benign samples 

print(f"Starting processing for dataset: {current_dataset_name.upper()}")
print("-" * 50)

# ==========================================
# 2. LOAD DATA
# ==========================================
big_columns = pd.read_csv(big_file, nrows=0).columns.tolist()
df = pd.read_csv(current_file_path)

df = df.rename(columns={
    'Fwd Pkts/s': 'Fwd Packets/s',
    'Bwd Pkts/s': 'Bwd Packets/s',
    'Bwd IAT Tot': 'Bwd IAT Total'
})

print(f"Original row count: {len(df)}")

# ❌ REMOVED NaN/Inf cleaning step as requested

# ==========================================
# 3. LABEL ROUTER
# ==========================================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()

known_benign_indicators = ["BENIGN", "0", "NORMAL"]
known_invalid_indicators = ["NONE", "NULL", "NAN", ""]

rows_before_trash = len(df)
df = df[~df['Label'].isin(known_invalid_indicators)]
dropped_invalid = rows_before_trash - len(df)

if dropped_invalid > 0:
    print(f"Cleaned Label count: {len(df)} (Dropped {dropped_invalid} corrupted labels)")

df_benign_source = df[df['Label'].isin(known_benign_indicators)]
df_attack_source = df[~df['Label'].isin(known_benign_indicators)]

# ==========================================
# 4. BENIGN PROCESSING (NOW USES m)
# ==========================================
if not df_benign_source.empty:
    benign_sample_size = min(m, len(df_benign_source))
    df_benign = df_benign_source.sample(n=benign_sample_size, random_state=42).copy()
    df_benign['Label'] = 0
else:
    df_benign = pd.DataFrame()
    benign_sample_size = 0

# ==========================================
# 5. ATTACK PROCESSING (UNCHANGED USING n)
# ==========================================
if not df_attack_source.empty:
    attack_classes = df_attack_source['Label'].unique()
    num_classes = len(attack_classes)
    target_per_class = n // num_classes

    print(f"Auto-detected {num_classes} distinct attack class(es).")
    print(f"Targeting {target_per_class} records per class.")

    attack_samples = []

    for attack_class in attack_classes:
        class_data = df_attack_source[df_attack_source['Label'] == attack_class]
        sample_size = min(target_per_class, len(class_data))
        attack_samples.append(class_data.sample(n=sample_size, random_state=42).copy())

    df_attack = pd.concat(attack_samples, ignore_index=True)
    df_attack['Label'] = 1
else:
    df_attack = pd.DataFrame()

# ==========================================
# 6. COMBINE & APPEND
# ==========================================
df_to_append = pd.concat([df_benign, df_attack], ignore_index=True)

if not df_to_append.empty:
    df_to_append = df_to_append.reindex(columns=big_columns)

    df_to_append.to_csv(big_file, mode='a', index=False, header=False)

    print("\nSUCCESS! Summary:")
    print(f"  -> Benign Rows: {benign_sample_size} (m-based)")
    print(f"  -> Attack Rows: {len(df_attack)} (n-based)")
    print(f"  -> Total Appended: {len(df_to_append)} rows")
else:
    print("\nWARNING: No valid records found to append!")

print("-" * 50)

Starting processing for dataset: CIC19
--------------------------------------------------
Original row count: 25031556
Auto-detected 5 distinct attack class(es).
Targeting 593 records per class.

SUCCESS! Summary:
  -> Benign Rows: 5933 (m-based)
  -> Attack Rows: 2811 (n-based)
  -> Total Appended: 8744 rows
--------------------------------------------------


In [2]:
import pandas as pd

df = pd.read_csv("leave_BoT-IoT.csv")
df['Label'].value_counts()

Label
0    35598
1    35436
Name: count, dtype: int64